# RecursiveGCD (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/RecursiveGCD_python.ipynb)

Chapter 14 — Recursion. Runnable notebook of `RecursiveGCD`: the algorithm in Python, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/RecursiveGCD_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/RecursiveGCD_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_python.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Python 3, nothing to install.

**Names.** There are no arrays in this chapter: variables and parameters have the names of the pseudo-code. Two exceptions in Python, where `from` and `next` have a special meaning: `Hanoi` uses `frm` instead of `from` and `IterativeFib` uses `nxt` instead of `next`.

## Tools: counters, stack and trace

The cell defines the counters (`cnt`, `calls`, `max_depth`), the call stack (`push` and `pop`) and the functions that print the trace (`event`, `say`) and the table of steps (`print_steps`). It has to be run only once.

In [1]:
cnt = [0] * 20      # cnt[r] = number of executions of line r (lines numbered as in the lecture notes)
calls = 0           # number of calls = nodes of the tree of recursive calls
max_depth = 0       # depth of the recursion = maximum height of the stack
mults = 0           # multiplications (FastPower)
moves = 0           # moves (Hanoi)
ev = 0              # counter of the events of the trace
stack = []          # the call stack: the arguments of the active calls
seen = [0] * 64     # seen[x] = number of calls with argument x
trace = False       # True: print the trace of the execution


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # a call starts: its frame goes on top of the stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # the call returns: its frame leaves the stack
    stack.pop()


def event(text):                   # numbered line of the trace, with the stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # line of the trace: the call, indented one level for every active call
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Line | Executions")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Total steps S = %d" % sum(cnt))

## Greatest common divisor: RecursiveGCD

**Input:** two natural numbers $n \ge 1$ and $m \ge 0$. **Output:** $\gcd(n, m)$. Tail recursion: each call makes at most one, the tree is a path. At most $2\lfloor\log_2 m\rfloor + 3$ calls, $O(\log_2 m)$ steps.

Pseudo-code: section 14.6 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
def floor_log2(m):                                 # floor(log2 m) with a loop (only for the bound on the calls)
    k = 0
    while m > 1:
        m = m // 2
        k = k + 1
    return k


def recursive_gcd(n, m):
    push(m); cnt[2] += 1
    if m == 0:
        cnt[3] += 1
        if trace: say("RecursiveGCD(%d, %d)" % (n, m), "base case, returns %d" % n)
        pop()
        return n
    if trace: say("RecursiveGCD(%d, %d)" % (n, m), "%d mod %d = %d: calls RecursiveGCD(%d, %d)" % (n, m, n % m, m, n % m))
    cnt[5] += 1
    g = recursive_gcd(m, n % m)
    if trace: say("RecursiveGCD(%d, %d)" % (n, m), "returns %d" % g)
    pop()
    return g

In [3]:
# Examples of the lecture notes: RecursiveGCD(48, 18) and RecursiveGCD(252, 105), trees of calls (paths)
for n, m in [(48, 18), (252, 105)]:
    reset()
    trace = True
    print("%-28s| %s" % ("call", "action"))
    g = recursive_gcd(n, m)
    trace = False
    print("gcd(%d, %d) = %d, calls = %d, S = %d" % (n, m, g, calls, sum(cnt)))
    print()

# Number of calls (each call executes 2 lines, hence S = 2 calls) and bound 2 floor(log2 m) + 3
print("      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3")
for n, m in [(48, 18), (252, 105), (89, 55), (144, 89), (1000, 7), (1000000, 999999)]:
    reset()
    g = recursive_gcd(n, m)
    print("%7d | %7d | %4d | %8d | %3d | %d" % (n, m, g, calls, sum(cnt), 2 * floor_log2(m) + 3))

call                        | action
RecursiveGCD(48, 18)        | 48 mod 18 = 12: calls RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: calls RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: calls RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | base case, returns 6
    RecursiveGCD(12, 6)     | returns 6
  RecursiveGCD(18, 12)      | returns 6
RecursiveGCD(48, 18)        | returns 6
gcd(48, 18) = 6, calls = 4, S = 8

call                        | action
RecursiveGCD(252, 105)      | 252 mod 105 = 42: calls RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: calls RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(42, 21)    | returns 21
  RecursiveGCD(105, 42)     | returns 21
RecursiveGCD(252, 105)      | returns 21
gcd(252, 105) = 21, calls = 4, S = 8

      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3
     48 |

**Try it yourself.** Change the two numbers; the worst case is two consecutive Fibonacci numbers.

In [4]:
n, m = 1071, 462        # <-- change the two numbers
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
g = recursive_gcd(n, m)
trace = False
print("gcd(%d, %d) = %d, calls = %d" % (n, m, g, calls))

call                        | action
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: calls RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: calls RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(147, 21)   | returns 21
  RecursiveGCD(462, 147)    | returns 21
RecursiveGCD(1071, 462)     | returns 21
gcd(1071, 462) = 21, calls = 4


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).